#### City Outing Forecast Prediction Tool
Purpose    : To give information about a city and predict the weather conditions so that a resident or traveller can go for outing \
LLM Method : We will use Anthropic "claude-sonnet-4-5" to take an input of city name, then provide details of top 5 attrractions, then call a tool to predict the weather on that day\
Input  : City Name and plan of travel with dates \
OutpUt : Top five attractions and current weather

This will help to understand how LLM can be used for calling tools externally to perform realworld actions

In [ ]:
import os
import json
from  dotenv import load_dotenv
from openai import OpenAI, api_key
import gradio as gr
from IPython.display import Markdown, display
from gradio.mcp import tool
from openai import base_url
import requests
from typer import params

In [ ]:
load_dotenv(override=True)
openai_key = os.getenv("OPENAI_API_KEY")
weather_key = os.getenv("OPEN_WEATHER_API_KEY")

In [ ]:
print(f"OpenAI Key       : {openai_key[:5]}")
print(f"Open Weather key : {weather_key[:5]}")

In [ ]:
openai=OpenAI()

In [ ]:
def get_weather_data(latitude, longitude):
    """
    :param latitude: Latitude, decimal (-90; 90). If you need the geocoder to automatic convert city names and zip-codes to geo coordinates and the other way around,
    :param longitude: Longitude, decimal (-180; 180). If you need the geocoder to automatic convert city names and zip-codes to geo coordinates and the other way around,
    :return: weather outlook
    """
    weather_url = "https://api.openweathermap.org/data/2.5/weather"
    params = {
        "lat" :round(float(latitude), 2),
        "lon" :round(float(longitude), 2),
        "exclude": "minutely,hourly,daily,alerts",
        "appid" : weather_key,
        "units": "metric"
    }
    try:
        response = requests.get(weather_url, params=params)
        data = response.json()
        return data
    except Exception as error:
        print(f"Error : while connecting to OpenWeather API : {error}")

In [ ]:
latitude = 55.86
longitude = 4.25
weather = get_weather_data(latitude, longitude)
weather
if weather:
    print(f"Current Temp: {weather['main']['temp']}°C")
    print(f"Weather: {weather['weather'][0]['description']}")
    print(f"Feels Like: {weather['main']['feels_like']}°C")
    print(f"Humidity: {weather['main']['humidity']}%")
    print(f"Visibility: {weather['visibility']} meters")

In [ ]:
get_weather_data = {
    "name": "get_weather_data",
    "description": "Use this tool to get weather data",
    "parameters": {
        "type": "object",
        "properties": {
            "latitude": {
                "type": "number",
                "description": "Latitude, decimal (-90; 90)."
            },
            "longitude": {
                "type": "number",
                "description": "Longitude, decimal (-180; 180)."
            },
            "city": {
                "type": "string",
                "description": "Name of the city customer is travelling to"
            }
        },
        "required": ["city"],
        "additionalProperties": False
    }
}

In [ ]:
tools = [{"type": "function", "function": get_weather_data }]

In [ ]:
tools

In [ ]:
system_prompt = ("""You are a travel agent trying to guide your customer to visit top 5 places in a city and its weather forecast \""
                 You will need to ask your customer the city they are traveling \
                 Then you can find latitude and longitude for that city and provide that back to the tool with function name get_weather_data""")

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history +  [{"role": "user", "content": message}]
    done = False
    while not done:
        model_name = "gpt-4o-mini"
        response = openai.chat.completions.create(
            model=model_name,
            messages=messages,
            tools=tools,l
        )
        finish_reason = response.choices[0].finish_reason
        if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = get_weather_data
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    return response.choices[0].message.content

In [42]:
gr.ChatInterface(chat).launch()


* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.
